In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# 1달치 목업 데이터 삽입 스크립트로 생성된 데이터 로드
df = pd.read_csv('output.csv')
print(f"데이터 크기: {df.shape[0]}행 x {df.shape[1]}열")
df.head()

데이터 크기: 30행 x 14열


,avg_dwell_time,core_customer_age,core_customer_gender,just_left_count,max_empty_table_time,max_response_wait_time,peak_time,temperature,total_count,captured_at,created_at,end_at,id,weather
0,70,30,1,0,18,4,19,19.4,52,2026-04-30 11:00:00.000000,2026-05-30 22:01:57.212084,2026-04-30 22:00:00.000000,1,SUNNY
1,67,30,1,1,12,7,18,19.8,35,2026-05-01 11:00:00.000000,2026-05-30 22:01:57.570298,2026-05-01 22:00:00.000000,2,SUNNY
2,68,30,1,1,32,4,19,18.5,59,2026-05-02 11:00:00.000000,2026-05-30 22:01:57.842495,2026-05-02 22:00:00.000000,3,CLOUDY
3,68,30,1,0,21,5,19,13.2,64,2026-05-03 11:00:00.000000,2026-05-30 22:02:01.373380,2026-05-03 22:00:00.000000,4,RAINY
4,72,30,1,3,19,4,18,16.1,37,2026-05-04 11:00:00.000000,2026-05-30 22:02:01.782206,2026-05-04 22:00:00.000000,5,CLOUDY


In [2]:
# 날짜형 변환
df['captured_at'] = pd.to_datetime(df['captured_at'])

# 요일 기반 주말 피처 생성 (월=0 ~ 일=6, 토/일 = 1)
df['day_of_week'] = df['captured_at'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# 시계열 지연 변수: 어제 실제 방문자 수 (Lag Feature)
df['prev_day_count'] = df['total_count'].shift(1)

# 첫째 날은 전날 카운트가 없어 NaN이 생기므로 삭제 처리
df_clean = df.dropna().reset_index(drop=True)

print(f"피처 가공 후 데이터 개수: {df_clean.shape[0]}개")
df_clean[['captured_at', 'is_weekend', 'prev_day_count', 'total_count']].head()

피처 가공 후 데이터 개수: 29개


,captured_at,is_weekend,prev_day_count,total_count
0,2026-05-01 11:00:00,0,52.0,35
1,2026-05-02 11:00:00,1,35.0,59
2,2026-05-03 11:00:00,1,59.0,64
3,2026-05-04 11:00:00,0,64.0,37
4,2026-05-05 11:00:00,0,37.0,38


In [3]:
# 범주형 데이터인 weather 원핫 인코딩 수행
df_encoded = pd.get_dummies(df_clean, columns=['weather'], drop_first=True)

# 실전 예측 시점에도 알 수 있는 유효 피처로만 입력 변수(X) 리스트 구성
features = ['temperature', 'is_weekend', 'prev_day_count']
features += [col for col in df_encoded.columns if col.startswith('weather_')]

X = df_encoded[features]
y = df_encoded['total_count']

print("최종 선정된 입력 피처 (X):", features)
X.head()

최종 선정된 입력 피처 (X): ['temperature', 'is_weekend', 'prev_day_count', 'weather_RAINY', 'weather_SUNNY']


,temperature,is_weekend,prev_day_count,weather_RAINY,weather_SUNNY
0,19.8,0,52.0,False,True
1,18.5,1,35.0,False,False
2,13.2,1,59.0,True,False
3,16.1,0,64.0,False,False
4,16.9,0,37.0,False,True


In [4]:
# 릿지 회귀의 규제가 모든 피처에 균등하게 작동하도록 StandardScaler 적용
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 80%는 학습, 20%는 검증 데이터셋으로 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"학습용 샘플 수: {X_train.shape[0]}개 | 검증용 샘플 수: {X_test.shape[0]}개")

학습용 샘플 수: 23개 | 검증용 샘플 수: 6개


In [5]:
# 릿지 회귀 모델 객체 선언 (규제 파라미터 alpha=1.0)
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# 테스트 데이터로 예측 및 성능 지표 계산
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=================== 📊 모델 평가 결과 ===================")
print(f"평균 절대 오차 (MAE): {mae:.2f} 명")
print(f"결정계수 (R² Score): {r2:.4f}")
print("==========================================================\n")

print("=================== 🔑 피처별 가중치 (Coefficients) ===================")
for feat, coef in zip(features, model.coef_):
    print(f" - {feat}: {coef:+.4f}")

=================== 📊 모델 평가 결과 ===================
평균 절대 오차 (MAE): 5.87 명
결정계수 (R² Score): 0.6841

=================== 🔑 피처별 가중치 (Coefficients) ===================
 - temperature: +0.8715
 - is_weekend: +8.3198
 - prev_day_count: -0.7832
 - weather_RAINY: +1.3463
 - weather_SUNNY: -0.1949


In [6]:
def predict_tomorrow(tomorrow_temp, tomorrow_weather, today_count, model, scaler, feature_columns):
    """
    내일의 온도 예보, 날씨 예보, 오늘의 실제 최종 정산 방문객 수를 입력받아
    내일 매장에 최종적으로 유입될 예상 방문자 수를 산출합니다.
    """
    # 내일 날짜 요일 기준 계산 (오늘 + 1일)
    tomorrow_date = pd.Timestamp.now() + pd.Timedelta(days=1)
    day_of_week = tomorrow_date.dayofweek
    is_weekend = 1 if day_of_week >= 5 else 0  # 토요일=5, 일요일=6
    
    # 기본 매핑 템플릿 생성
    input_data = {
        'temperature': tomorrow_temp,
        'is_weekend': is_weekend,
        'prev_day_count': today_count
    }
    
    # 원핫 인코딩 날씨 매핑
    for col in feature_columns:
        if col.startswith('weather_'):
            weather_type = col.replace('weather_', '')
            input_data[col] = 1 if tomorrow_weather.upper() == weather_type.upper() else 0
            
    # 피처 컬럼 정렬 일치화
    input_df = pd.DataFrame([input_data])[feature_columns]
    
    # 학습용 스케일러 그대로 변환 적용
    input_scaled = scaler.transform(input_df)
    
    # 예측 수행 및 후처리
    pred_raw = model.predict(input_scaled)[0]
    final_pred = max(0, int(round(pred_raw)))
    
    kor_days = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]
    
    print(f"🔮 [SPOTLINE AI 내일 예측 보고서]")
    print(f"  - 예측 기준 일자: {tomorrow_date.strftime('%Y-%m-%d')} ({kor_days[day_of_week]})")
    print(f"  - 내일 최고 기온: {tomorrow_temp} ℃")
    print(f"  - 내일 기상 상태: {tomorrow_weather}")
    print(f"  - 오늘 최종 방문자수: {today_count} 명")
    print(f"--------------------------------------------------")
    print(f"👉 내일 삼겹살집 예상 방문객 수: 【 {final_pred} 명 】")
    return final_pred

# === [가상 시뮬레이션 예시] ===
# 내일 최고 기온이 21.5도, 날씨가 비가 올 예정(RAINY), 오늘 매장 정산 고객이 42명일 때의 가상 상황 예측
print("[시뮬레이션 1. 비 내리는 평일]")
predict_tomorrow(
    tomorrow_temp=21.5, 
    tomorrow_weather='RAINY', 
    today_count=42, 
    model=model, 
    scaler=scaler, 
    feature_columns=features
)

[시뮬레이션 1. 비 내리는 평일]
🔮 [SPOTLINE AI 내일 예측 보고서]
  - 예측 기준 일자: 2026-06-01 (월요일)
  - 내일 최고 기온: 21.5 ℃
  - 내일 기상 상태: RAINY
  - 오늘 최종 방문자수: 42 명
--------------------------------------------------
👉 내일 삼겹살집 예상 방문객 수: 【 46 명 】


46